In [1]:
from typing_extensions import Literal, TypedDict
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from typing import Annotated, List
import operator
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_community.utilities import GoogleSerperAPIWrapper
import pprint
import json


In [2]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"]=os.getenv("LANGSMITH_PROJECT")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["SERPER_API_KEY"] = os.getenv("SERPER_API_KEY")


In [3]:
llm = ChatGroq(model="gemma2-9b-it")

In [4]:
class Section(BaseModel):
    title: str = Field(
        description="A title of the news",
    )
    link: str = Field(
        description="A link to the news",
    )
    time_published: str = Field(
        description="A time when news published",
    )
    source: str = Field(
        description="A news chennel name",
    )
    # snippet: str = Field(
    #     description="News overview",
    # )

class Sections(BaseModel):
    sections: List[Section] = Field(
        description="Sections of the AI news",
    )

class query(BaseModel):
    news_topic: str = Field(description="string for topic")


class NewsItem(BaseModel):
    title: str
    link: str
    date: str
    source: str

class RelevantNews(BaseModel):
    results: List[NewsItem]

relevant_news_agent = llm.with_structured_output(RelevantNews)

# Augment the LLM with schema for structured output
planner = llm.with_structured_output(Sections)
newsai = llm.with_structured_output(query)


In [5]:
orchestration_instruction = """
# Orchestrator Agent Instructions

**Role:**  
You are the Orchestrator Agent in a multi-agent AI system for curating and summarizing AI-related news.

**Objective:**  
Analyze a list of news items and generate a **structured outline (plan)** for a news report.

**Input Format:**  
You will receive the latest AI news as a list of dictionaries. Each news item includes:
- **title**
- **link**
- **date** (time published)
- **source**

**Tasks:**  
1. **Analyze:** Carefully review each news item.
2. **Organize:** Arrange the news into a clear, well-structured outline. Group items by category or relevance if possible.
3. **Output:** Produce a well-organized outline (plan) based solely on the provided news information.

Return only the **structured outline** as your final output.
"""

worker_instruction = """
# Worker Agent Instructions

**Role:**  
You are a Worker Agent in news summarization system tasked with transforming a single news item into an engaging and insightful summary. make sure heading should align properly.

**Steps to Follow:**

2. **Title:**  
   - Provide a compelling and relevant title for the news.

3. **Source & Timestamp:**  
   - Immediately below the title, include the news source and publication time.

4. **Article Understanding:**  
   - Use the provided link to read and fully understand the article.

5. **Snippet:**  
   - Based on article understanding, Write a brief, engaging and factual snippet that captures the essence of the news.

6. **Summary:**  
   - Offer a concise 3-5 sentence summary that covers the key points.

7. **Analysis:**  
   - Provide a short analysis discussing the article's significance, its relation to past developments, and its potential future impact.

8. **Key Insights:**  
   - List 2-3 bullet points highlighting the most important takeaways.

9. **Original Source:**  
   - Conclude with the direct link to the original article.

**Formatting Requirements:**  
- Use clean **Markdown formatting** with clear headings and highlights.
- Do not include any additional commentary or extraneous text.
"""

news_instruction = """
You are an intelligent news-search agent. Your role is to deeply understand the user's query and generate a concise, information-rich search topic that can retrieve all relevant news articles from google serper.
below is the code where we will pass topic. topic should be string so you just have to give a small string.

def news(topic: str):
    search = GoogleSerperAPIWrapper(type="news", tbs="qdr:h24")
    results = search.results(topic)

"""

verification_instruction = """
You are an AI assistant named VerifiNews. Your role is to carefully examine each news dictionary in a list passed to you.

Each dictionary has the fields: `title`, `link`, `date`, and `source`.

Keep only those dictionaries that:
- Have a complete and meaningful `title`
- Have a valid `link` 
- Have a `date` in correct format 
- Have a valid `source`

Remove any news dictionary that is:
- duplicate
- Incomplete
- Has junk or empty values
- Has a broken or invalid URL
- Has an incorrect or missing date

Return only the cleaned list of valid news dictionaries. Do not return anything else.
"""

analysis_instruction = """
You are a knowledge analyst agent with access to the web.

TASK:
Given the title, link, publication date, and source of a news article, your job is to:

1. **Fetch the article content** from the provided URL.3. Immediately below the title, include the news source and publication time.

2. Conduct an **in-depth factual analysis**, including:
   - Summary of the article (clear, concise, accurate)
   - Business or technological insights (if applicable)
   - Psychological, societal, or philosophical implications (if relevant)
   - Future impact or long-term significance
3. Identify the article's **relevance** to one or more of these domains:
   - Artificial Intelligence (AI)
   - Environment and Climate
   - Economics and Investing
   - Ethics, Philosophy, or Psychology
4. Provide a well-structured final output with the following sections:
   - Summary
   - Key Insights
   - Broader Implications
   - Future Impact
   - Relevance
   - Conclusion

5. **Original Source:**  
   - Conclude with the direct link to the original article.

RULES:
- Stick to **factual information** from the article and reliable sources.
- If the link is broken or the article can't be fetched, report that clearly.
- Do not fabricate content.
- Use markdown for formatting.

INPUT:
- Title: {title}
- Link: {link}
- Date: {date}
- Source: {source}
"""

explainer_instructions = """
You are Explainer, an intelligent helpful assistant.
Answer the users question acurately in short.
Provide factual answers.
"""

In [7]:
import feedparser
from datetime import datetime, timedelta, timezone
import requests

# RSS feeds organized by type
rss_feeds = {
    "Top Stories": [
        "https://www.cbc.ca/cmlink/rss-topstories",
        "https://www.thestar.com/search/?f=rss&t=article&bl=2827101&l=20",
        "https://globalnews.ca/feed/",
        "https://nationalpost.com/feed/",
    ],
    "World News": [
        "https://www.cbc.ca/cmlink/rss-world",
        "https://globalnews.ca/world/feed/",
        "http://rss.cnn.com/rss/edition_world.rss"
    ],
    "Canada News": [
        "https://www.cbc.ca/cmlink/rss-canada",
        "https://globalnews.ca/canada/feed/"
    ],
    "Business": [
        "https://www.cbc.ca/cmlink/rss-business",
        "https://www.theglobeandmail.com/report-on-business/?service=rss",
        "https://business.financialpost.com/feed/"
    ],
    "Technology": [
        "https://www.cbc.ca/cmlink/rss-technology",
        "https://www.theglobeandmail.com/technology/?service=rss"
    ],
    "Politics": [
        "https://globalnews.ca/politics/feed/",
        "https://www.cbc.ca/webfeed/rss/rss-politics"
    ],
    "Health": [
        "https://globalnews.ca/health/feed/",
        "https://www.cbc.ca/webfeed/rss/rss-health"
    ]
}

# Time setup for 24-hour filtering
now = datetime.now(timezone.utc)
print(now)
print(timedelta(days=1))
last_24hrs = now - timedelta(days=1)
print(last_24hrs)

# Final result list
all_news = []
print(rss_feeds.items())


2025-03-31 22:49:48.072074+00:00
1 day, 0:00:00
2025-03-30 22:49:48.072074+00:00
dict_items([('Top Stories', ['https://www.cbc.ca/cmlink/rss-topstories', 'https://www.thestar.com/search/?f=rss&t=article&bl=2827101&l=20', 'https://globalnews.ca/feed/', 'https://nationalpost.com/feed/']), ('World News', ['https://www.cbc.ca/cmlink/rss-world', 'https://globalnews.ca/world/feed/', 'http://rss.cnn.com/rss/edition_world.rss']), ('Canada News', ['https://www.cbc.ca/cmlink/rss-canada', 'https://globalnews.ca/canada/feed/']), ('Business', ['https://www.cbc.ca/cmlink/rss-business', 'https://www.theglobeandmail.com/report-on-business/?service=rss', 'https://business.financialpost.com/feed/']), ('Technology', ['https://www.cbc.ca/cmlink/rss-technology', 'https://www.theglobeandmail.com/technology/?service=rss']), ('Politics', ['https://globalnews.ca/politics/feed/', 'https://www.cbc.ca/webfeed/rss/rss-politics']), ('Health', ['https://globalnews.ca/health/feed/', 'https://www.cbc.ca/webfeed/rss/rs

In [8]:
last_24hrs

datetime.datetime(2025, 3, 30, 22, 49, 48, 72074, tzinfo=datetime.timezone.utc)

checkout difference b/w times.

In [9]:
# fetch_rss.py (with timeout + fallback)
def rss_news():
    rss_feeds_news = []
    for urls in rss_feeds.values():
            for url in urls:
                try:
                    # Fetch RSS with timeout
                    response = requests.get(url, timeout=5)
                    feed = feedparser.parse(response.content)

                    for entry in feed.entries:
                        # print(entry)
                        try:
                            published_time = datetime(*entry.published_parsed[:6])
                            published_time = published_time.replace(tzinfo=timezone.utc)
                            print("published_time", published_time)
                            if published_time > last_24hrs:
                                # print(entry.title)
                                news_item = {
                                    "title": entry.title,
                                    "link": entry.link,
                                    "date": published_time.strftime('%Y-%m-%d %H:%M:%S'), #todo
                                    "source": feed.feed.get("title", "Unknown Source"), #todo
                                }
                                rss_feeds_news.append(news_item)
                        except Exception as e:
                            print(f"⚠️ Entry error: {e}")
                            continue

                except requests.exceptions.Timeout:
                    print(f"⏰ Timeout: {url}")
                except Exception as e:
                    print(f"❌ Failed to fetch {url}: {e}")

    return rss_feeds_news


DB

In [10]:
# db_handler.py
import sqlite3
from datetime import datetime, timedelta, timezone

DB_NAME = "news_ai.db"

In [11]:
# Initialize database and table
def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS news (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            link TEXT UNIQUE,
            date TEXT,
            source TEXT,
            fetched_at       
        )
    ''')
    conn.commit()
    conn.close()

# Insert news into database (skip duplicates)
def insert_news(news_list):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    now = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
    print("insert start")
    for news in news_list:
        try:
            cursor.execute('''
                INSERT OR IGNORE INTO news (title, link, date, source, fetched_at)
                VALUES (?, ?, ?, ?, ?)
            ''', (news["title"], news["link"], news["date"], news["source"], now))
        except Exception:
            continue
    print("insert done")
    conn.commit()
    conn.close()


# Delete news older than X hours (default: 48 hours)
def delete_old_news(hours=48):
    threshold_time = (datetime.now(timezone.utc) - timedelta(hours=hours)).strftime('%Y-%m-%d %H:%M:%S')
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM news WHERE fetched_at < ?", (threshold_time,))
    conn.commit()
    conn.close()


# Get news by optional filters (e.g., keyword only)
def get_rss_news(keyword=None, limit=100):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    print(keyword)
    query = "SELECT title, link, date, source FROM news WHERE 1=1"
    params = []

    if keyword:
        query += " AND LOWER(title) LIKE ?"
        params.append(f"%{keyword.lower()}%")

    query += " ORDER BY date DESC LIMIT ?"
    params.append(limit)

    cursor.execute(query, params)
    results = cursor.fetchall()
    conn.close()

    return [
        {"title": r[0], "link": r[1], "date": r[2], "source": r[3]}
        for r in results
    ]

def chunk_news(news_list, size=40):
    return [news_list[i:i+size] for i in range(0, len(news_list), size)]


# def get_rss_news(news_list, user_query):
#     relevant_chunks = []

#     for chunk in chunk_news(news_list):
#         prompt = f"""
#         User asked: "{user_query}"

#         Here is a list of news articles (title, date, source, link):

#         {json.dumps(chunk, indent=2)}

#         Please extract only the articles that are relevant to the user's query.
#         Return them as a JSON list of dictionaries.
#         """
#         result = llm.predict(prompt)
#         filtered = json.loads(result)
#         relevant_chunks.extend(filtered)

In [ ]:
news_list

In [350]:
size = 40
chunk = [news_list[i:i+size] for i in range(0, len(news_list), size)]

In [ ]:
chunk

In [ ]:
user_query = "Pierre Poilievre"
relevant_chunks = []
for chunk in chunk:
        prompt = f"""
        User asked: "{user_query}"

        Here is a list of news articles (title, date, source, link):

        {json.dumps(chunk, indent=2)}

        Please extract only the articles that are relevant to the user's query.
        Return them as a JSON list of dictionaries.
        """
        result = relevant_news_agent.invoke(prompt)
        print(result.results)
        cleaned = [item.model_dump() for item in result.results]

        
        relevant_chunks.extend(cleaned)

relevant_chunks

In [12]:
init_db()

In [13]:
news_list = rss_news()

⏰ Timeout: https://www.cbc.ca/cmlink/rss-topstories
published_time 2025-03-31 22:41:00+00:00
published_time 2025-03-31 20:30:00+00:00
published_time 2025-03-31 21:45:00+00:00
published_time 2025-03-31 15:15:00+00:00
published_time 2025-03-31 15:21:00+00:00
published_time 2025-03-31 18:00:00+00:00
published_time 2025-03-31 21:45:00+00:00
published_time 2025-03-31 09:00:00+00:00
published_time 2025-03-31 15:57:00+00:00
published_time 2025-03-31 09:00:00+00:00
published_time 2025-03-31 19:50:00+00:00
published_time 2025-03-31 18:45:00+00:00
published_time 2025-03-31 15:27:00+00:00
published_time 2025-03-31 06:59:00+00:00
published_time 2025-03-31 17:30:00+00:00
published_time 2025-03-31 20:25:05+00:00
published_time 2025-03-31 12:45:00+00:00
published_time 2025-03-31 17:53:00+00:00
published_time 2025-03-31 06:46:00+00:00
published_time 2025-03-31 09:00:00+00:00
published_time 2025-03-31 22:49:10+00:00
published_time 2025-03-31 22:27:23+00:00
published_time 2025-03-31 22:20:55+00:00
publi

In [14]:
news_list

[{'title': "Most supervised consumption sites that were set to close won't offer service despite injunction. Here's why",
  'link': 'https://www.thestar.com/news/gta/most-supervised-consumption-sites-that-were-set-to-close-wont-offer-service-despite-injunction-heres/article_92086a33-1354-4fe7-9c44-ab2d9fecfb7f.html',
  'date': '2025-03-31 22:41:00',
  'source': 'www.thestar.com - RSS Results of type article'},
 {'title': "Meet Pierre Poilievre's not-so-secret weapon: his wife",
  'link': 'https://www.thestar.com/politics/federal/meet-pierre-poilievres-not-so-secret-weapon-his-wife/article_32a27b18-da7a-4834-a899-096904c65ee5.html',
  'date': '2025-03-31 20:30:00',
  'source': 'www.thestar.com - RSS Results of type article'},
 {'title': "'Frustrated and mad': Conservative insiders say Pierre Poilievre is fumbling the campaign by not focusing on Donald Trump",
  'link': 'https://www.thestar.com/politics/federal/frustrated-and-mad-conservative-insiders-say-pierre-poilievre-is-fumbling-the

In [15]:
insert_news(news_list=news_list)

insert start
insert done


In [16]:
delete_old_news()

In [17]:
from langgraph.constants import Send
from typing import Annotated, Sequence
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class ExplainerState(TypedDict):
    question: str
    answer: str

# Graph state
class State(TypedDict):
    news_topic: str  # news topic
    latest_news: list[dict]
    sections: list[Section]  # List of news sections
    completed_sections: Annotated[list, operator.add]  # All workers write to this key in parallel
    final_report: str  # Final report

# Worker state
class WorkerState(TypedDict):
    section: Section
    completed_sections: Annotated[list, operator.add]

# Nodes
def google_news(topic: str):
    search = GoogleSerperAPIWrapper(type="news", tbs="qdr:h24")
    results = search.results(topic)
    #pprint.pp(results)
    
    for news in results["news"]:
        for key in ['imageUrl', 'position', 'snippet']:
            if key in news:
                del news[key] 

    #pprint.pp(results)
    return results['news'] 

def news(topic: str):
    g_news = google_news(topic)
    r_news = get_rss_news(keyword=topic)
    print("r_news = ",r_news)
    latest_news = g_news + r_news
    return latest_news

def news_ai(state: State):
    """AI agent that understand user query and generate string for news extraction"""

    news_topic = newsai.invoke(
        [
            SystemMessage(content=news_instruction),
            HumanMessage(
                content=f"Here is the user query: {state['news_topic']}"
            ),
        ]
    )
    #news_topic_dict = news_topic.model_dump()
    # print("news_topic:", news_topic_dict)

    topic = news_topic.news_topic
    print(topic)
    # g_news = google_news(topic)
    # r_news = get_rss_news(keyword=topic)
    # latest_news = g_news + r_news
    latest_news = news(topic=topic)
    print("latest =", latest_news)
    return {"latest_news": latest_news}
    
def verify_news(state: State):
    verified_news = llm.invoke(
        [
            SystemMessage(content=verification_instruction),
            HumanMessage(
                content=f"Here is all the news details: {state['latest_news']}"
            ),
        ]
    )
    print("verified = ", verified_news)
    return {"latest_news": verified_news}


def orchestrator(state: State):
    """Orchestrator that generates a plan for the news"""

    #print("Orchestrator is activated, here output will be lst of section: State")
    # latest_news = news(state["news_topic"])
    # print(latest_news)
    # Generate queries
    report_sections = planner.invoke(
        [
            SystemMessage(content=orchestration_instruction),
            HumanMessage(
                content=f"Here is the latest AI news: {state['latest_news']}"
            ),
        ]
    )
    #print(state["topic"])
    print("Report Sections:",report_sections)

    return {"sections": report_sections.sections}

def llm_call(state: WorkerState):
    """Worker writes a section of the report"""

    #print("llm_call is activated, here output will be lst of section: WorkerState, section = ")
    # print(f"state[section] = {state['section']}")
    # Generate section
    section = llm.invoke(
        [
            SystemMessage(content=analysis_instruction),
            HumanMessage(
                content=f"Here is the news_title: {state['section'].title},\
                      news_link: {state['section'].link}, time_published: {state['section'].time_published},\
                          news_source: {state['section'].source}" # news_snippet: {state['section'].snippet}
            ),
        ]
    )

    #print(f"{state['section']},\n {type(state['section'])}")
    # Write the updated section to completed sections
    #print("section = \n", section)
    return {"completed_sections": [section.content]}


def synthesizer(state: State):
    """Synthesize full report from sections"""

    #print("synthesizer is activated, here output will be lst of completed sections: State")

    # List of completed sections
    completed_sections = state["completed_sections"]
    #print("completed_sections",completed_sections)
    # Format completed section to str to use as context for final sections
    completed_report_sections = "\n\n---\n\n".join(completed_sections)

    #print("completed_report_sections",completed_report_sections)

    return {"final_report": completed_report_sections}


# Conditional edge function to create llm_call workers that each write a section of the report
def assign_workers(state: State):
    """Assign a worker to each section in the plan"""
    #print([s for s in state["sections"]])
    # Kick off section writing in parallel via Send() API
    return [Send("llm_call", {"section": s}) for s in state["sections"]]

def explainer(state: ExplainerState):
    
    # messages = state["messages"]

    # question = messages[0].content
    explainer = llm
    response = explainer.invoke(
        [
            SystemMessage(content=explainer_instructions),
            HumanMessage(
                content=f"Here is the question: {state['question']}"
            ),
        ]
    )
   
    # print(state["question"])
    # print("response: ", response)
 
    return {"answer": response.content}

In [ ]:
news("israel")

In [18]:
# Build workflow
builder = StateGraph(State)
builder_2 = StateGraph(ExplainerState)

# Add the nodes
builder.add_node("news_ai", news_ai)
# builder.add_node("verify_news", verify_news)
builder.add_node("orchestrator", orchestrator)
builder.add_node("llm_call", llm_call)
builder.add_node("synthesizer", synthesizer)
builder_2.add_node("explainer", explainer)

# Add edges to connect nodes
builder.add_edge(START, "news_ai")
builder.add_edge("news_ai", "orchestrator")
# builder.add_edge("news_ai", "verify_news")
# builder.add_edge("verify_news", "orchestrator")
builder.add_conditional_edges(
    "orchestrator", assign_workers, ["llm_call"]
)
builder.add_edge("llm_call", "synthesizer")
builder.add_edge("synthesizer", END)

# new graph..
builder_2.add_edge(START, "explainer")
builder_2.add_edge("explainer", END)

# Compile the workflow
graph = builder.compile()
graph_2 = builder_2.compile()

graph, graph_2

(<langgraph.graph.state.CompiledStateGraph at 0x1aba1ad2290>,
 <langgraph.graph.state.CompiledStateGraph at 0x1aba1ad3e80>)

In [19]:
# Invoke
response = graph.invoke({"news_topic": "Pierre Poilievre"})

Pierre Poilievre
Pierre Poilievre
r_news =  [{'title': "'Frustrated and mad': Conservative insiders say Pierre Poilievre is fumbling the campaign by not focusing on Donald Trump", 'link': 'https://www.thestar.com/politics/federal/frustrated-and-mad-conservative-insiders-say-pierre-poilievre-is-fumbling-the-campaign-by-not-focusing/article_e3dc95bf-6067-49b5-a037-3d6e099712c3.html', 'date': '2025-03-31 21:45:00', 'source': 'www.thestar.com - RSS Results of type article'}, {'title': "Meet Pierre Poilievre's not-so-secret weapon: his wife", 'link': 'https://www.thestar.com/politics/federal/meet-pierre-poilievres-not-so-secret-weapon-his-wife/article_32a27b18-da7a-4834-a899-096904c65ee5.html', 'date': '2025-03-31 20:30:00', 'source': 'www.thestar.com - RSS Results of type article'}, {'title': "Race tightening between Mark Carney's Liberals and Pierre Poilievre's Conservatives, polls suggest", 'link': 'https://www.thestar.com/politics/federal/race-tightening-between-mark-carneys-liberals-an

In [20]:
response

{'news_topic': 'Pierre Poilievre',
 'latest_news': [{'title': "Mark Carney's Chances of Beating Pierre Poilievre in Canada: Recent Polls",
   'link': 'https://www.newsweek.com/mark-carney-chances-beating-pierre-poilievre-canada-polls-2053024',
   'date': '7 hours ago',
   'source': 'Newsweek'},
  {'title': "Poilievre says the federal election can't just be about Donald Trump",
   'link': 'https://www.cbc.ca/news/politics/poilievre-campaign-messaging-1.7497965',
   'date': '6 hours ago',
   'source': 'CBC'},
  {'title': "Pierre Poilievre dismisses calls for 'pivot' as Liberals increase lead in polls",
   'link': 'https://nationalpost.com/news/politics/federal_election/pierre-poilievre-election-strategy-liberal-polls',
   'date': '2 hours ago',
   'source': 'National Post'},
  {'title': 'Poilievre pledges ‘national energy corridor’ to expedite pipelines, infrastructure',
   'link': 'https://www.theglobeandmail.com/politics/federal-election/article-poilievre-pledges-national-energy-corrid

In [ ]:
from IPython.display import Markdown
Markdown(response["final_report"])

In [285]:
response = graph_2.invoke({"question": "omlet"})

In [286]:
response["answer"]

'An omelette is a dish made from beaten eggs that are fried in butter or oil and typically filled with cheese, vegetables, or meat.  \n'

In [287]:
from IPython.display import Markdown
Markdown(response["answer"])

An omelette is a dish made from beaten eggs that are fried in butter or oil and typically filled with cheese, vegetables, or meat.  


# In depth Analysis

In [288]:
analyser = ChatGroq(model="gemma2-9b-it")

In [289]:
news = {'title': 'Philadelphia Phillies vs Washington Nationals: Where and how to watch today’s match, venue, time and more',
   'link': 'https://timesofindia.indiatimes.com/sports/mlb/news/philadelphia-phillies-vs-washington-nationals-where-and-how-to-watch-todays-match-venue-time-and-more/articleshow/119761771.cms',
   'date': '5 hours ago',
   'source': 'Times of India'}

Indepth analyzer

In [290]:
analysis = analyser.invoke(
        [
            SystemMessage(content=analysis_instruction),
            HumanMessage(
                content=f"Here is the news details: {news}"
            ),
        ]
    )

In [291]:
analysis.content

'## Philadelphia Phillies vs Washington Nationals: Where and how to watch today’s match\n\n**Source:** Times of India  \n**Publication Time:** 5 hours ago\n\n### Summary\n\nThis article provides details on how to watch the Philadelphia Phillies vs. Washington Nationals baseball game. It includes the venue (Nationals Park in Washington D.C.), the start time (7:05 PM EDT), and the broadcasting information (ESPN, MLB.TV). \n\n### Key Insights\n\nThe article highlights the specific platforms where fans can access the game, catering to both traditional television viewers and those who prefer streaming.  \n\n### Broader Implications\n\nThe article reflects the growing interest in baseball, particularly in accessing live games through various media channels. \n\n### Future Impact\n\nThe continued availability of live games across multiple platforms is likely to further enhance fan engagement and potentially contribute to the growth of baseball viewership.\n\n### Relevance\n\n* **Sports:** Thi

In [292]:
from IPython.display import Markdown
Markdown(analysis.content)

## Philadelphia Phillies vs Washington Nationals: Where and how to watch today’s match

**Source:** Times of India  
**Publication Time:** 5 hours ago

### Summary

This article provides details on how to watch the Philadelphia Phillies vs. Washington Nationals baseball game. It includes the venue (Nationals Park in Washington D.C.), the start time (7:05 PM EDT), and the broadcasting information (ESPN, MLB.TV). 

### Key Insights

The article highlights the specific platforms where fans can access the game, catering to both traditional television viewers and those who prefer streaming.  

### Broader Implications

The article reflects the growing interest in baseball, particularly in accessing live games through various media channels. 

### Future Impact

The continued availability of live games across multiple platforms is likely to further enhance fan engagement and potentially contribute to the growth of baseball viewership.

### Relevance

* **Sports:** This article is clearly relevant to the sports domain, specifically baseball. 

### Conclusion

The article serves as a convenient guide for fans interested in watching the Phillies vs. Nationals game. 

**Original Source:** https://timesofindia.indiatimes.com/sports/mlb/news/philadelphia-phillies-vs-washington-nationals-where-and-how-to-watch-todays-match-venue-time-and-more/articleshow/119761771.cms
